# Partitioning articles by topic

## Import statements

In [1]:
from bertopic import BERTopic
import pandas as pd
import os
import numpy as np

/Users/ameliemajor/miniforge3/envs/bertopic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import plotly.io as pio
pio.renderers.default = "browser"


## Load dataframe and filter by `bodyContent`

In [3]:
df = pd.read_csv('guardian_combined.csv')
df.head()
df = df.copy()

In [4]:


# Keep ONLY rows with real text for BERTopic (preserve index for safe merge-back)

bodyText = df["content"].astype(str).tolist()

print(f"Full DataFrame shape: {df.shape}")
print(f"Rows used for BERTopic: {df.shape}")
print(f"Texts passed to BERTopic: {len(bodyText)}")


Full DataFrame shape: (27575, 7)
Rows used for BERTopic: (27575, 7)
Texts passed to BERTopic: 27575


## Fit BERTopic to `content`

In [5]:
from umap import UMAP
from hdbscan import HDBSCAN

RANDOM_SEED = 42

umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
)

print(f"Fitting the BERTopic model (random_seed={RANDOM_SEED})...")
topics, probs = topic_model.fit_transform(bodyText)
df["topic_id"] = np.nan
df.loc[df.index, "topic_id"] = topics
df["topic_id"] = df["topic_id"].astype("Int64")

Fitting the BERTopic model (random_seed=42)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1410.23it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Visualize findings

In [6]:
topic_model.visualize_barchart(top_n_topics=10)

In [7]:
topic_model.visualize_topics()

In [8]:
topic_model.get_topic_info(10)

,Topic,Count,Name,Representation,Representative_Docs
0,10,193,10_facebook_zuckerberg_google_users,"[facebook, zuckerberg, google, users, data, te...","[Last week, Facebook released a set of colorfu..."


## Select non-person-entity topics
e.g. not a politician as a topic

In [9]:
# topic_id has already been assigned safely using the fitted subset's index.
# (Do NOT assign topic_model.topics_ directly to the full dataframe; lengths can differ.)
assert "topic_id" in df.columns, "topic_id column missing—run the BERTopic fitting cell first."
print("topic_id assigned:", df["topic_id"].notna().sum(), "rows; missing:", df["topic_id"].isna().sum())
political_guardian_articles_df = df


topic_id assigned: 27575 rows; missing: 0


## Write to files

In [10]:
os.makedirs("articles_by_topic", exist_ok=True)

# Build topic_name from BERTopic's own labels (top words joined by underscores)
topic_info = topic_model.get_topic_info()
id_to_name = dict(zip(topic_info["Topic"], topic_info["Name"]))

df["topic_name"] = df["topic_id"].map(id_to_name).fillna("Outlier")

# Sanitise names for use as filenames
def to_filename(name):
    return "".join(c if c.isalnum() or c in " _-" else "_" for c in str(name)).strip().replace(" ", "_")

# Top 10 topics by article count (excluding outlier topic -1)
top10_ids = (
    topic_info[topic_info["Topic"] != -1]
    .nlargest(10, "Count")["Topic"]
    .tolist()
)

written = []
for topic_id in top10_ids:
    name = id_to_name[topic_id]
    group = df[df["topic_id"] == topic_id]
    filename = f"articles_by_topic/{to_filename(name)}.csv"
    group.to_csv(filename, index=False)
    written.append((topic_id, name, len(group)))

written.sort(key=lambda x: -x[2])
print(f"Wrote {len(written)} topic files (top 10 by article count):")
for tid, name, count in written:
    print(f"  topic {str(tid):<4}  {count:>5} articles  →  {to_filename(name)}.csv")

political_guardian_articles_df = df

Wrote 10 topic files (top 10 by article count):
  topic 0       596 articles  →  0_covid_virus_vaccine_pandemic.csv
  topic 1       562 articles  →  1_climate_emissions_carbon_energy.csv
  topic 2       474 articles  →  2_scottish_scotland_snp_sturgeon.csv
  topic 3       243 articles  →  3_biden_trump_sanders_democratic.csv
  topic 4       235 articles  →  4_schools_school_education_teachers.csv
  topic 5       221 articles  →  5_farage_ukip_nuttall_party.csv
  topic 6       202 articles  →  6_her_she_may_brexit.csv
  topic 7       199 articles  →  7_book_books_novel_my.csv
  topic 8       198 articles  →  8_nhs_health_doctors_patients.csv
  topic 9       196 articles  →  9_eu_barnier_uk_deal.csv
